# Document-level legal reference linking - showcase

}
        {
            "cell_type": "code",
            "metadata": {
                "language": "python"
            },
            "source": [
                "still_unresolved = [",
                "    outcome[\"reference\"]",
                "    for outcome in recheck_outcomes",
                "    if outcome[\"status\"] == \"unresolved\"",
                "]",
                "",
                "print(f\"All still-unresolved references ({len(still_unresolved)}):\")",
                "for index, reference in enumerate(still_unresolved, start=1):",
                "    print(f\"{index}. {reference}\")"
            ]
        }
render linked HTML. This notebook showcases the document-level service in
`docs/Technical_Design_Document_v3.md`; mandatory mention offsets are validated
against the normalized text before rendering.

## Setup


In [ ]:
import importlib
import json
import logging
from pathlib import Path

logging.basicConfig(level=logging.WARNING, format="%(levelname)s %(name)s: %(message)s")
for logger_name in (
    "httpx",
    "httpx2",
    "httpcore",
    "mcp.client.streamable_http",
):
    logging.getLogger(logger_name).setLevel(logging.WARNING)
for logger_name in (
    "src.document_service",
    "src.agent_helpers.search_graph",
    "src.agent_helpers.search_execution",
    "src.document_helpers.cache",
):
    logging.getLogger(logger_name).setLevel(logging.INFO)

import src.agent as agent_module
import src.agent_helpers.models as models_module
import src.agent_helpers.search_execution as search_execution_module
import src.agent_helpers.search_graph as search_graph_module
import src.config as config_module
import src.document_helpers.aggregation as aggregation_module
import src.document_helpers.cache as cache_module
import src.document_helpers.extraction as extraction_module
import src.document_helpers.rendering as rendering_module
import src.document_service as document_service_module

importlib.reload(models_module)
importlib.reload(search_execution_module)
importlib.reload(search_graph_module)
importlib.reload(agent_module)
importlib.reload(config_module)
importlib.reload(extraction_module)
importlib.reload(cache_module)
importlib.reload(aggregation_module)
importlib.reload(rendering_module)
importlib.reload(document_service_module)

from src.agent import Agent
from src.agent_helpers.models import DocumentContext
from src.document_service import DocumentService
from src.document_helpers.rendering import fetch_formatted_page, read_linked_html, write_linked_html

agent_config = config_module.load_agent_config()
agent = Agent(agent_config)
document_service = DocumentService(agent_config, agent=agent)


async def find_link_to_document(
    reference: str,
    document_context: DocumentContext | None = None,
):
    return await agent.search(reference, document_context=document_context)


def format_resolution(response, full: bool = False):
    print("=" * 80)
    print("RESOLUTION SUMMARY")
    print("=" * 80)
    print(f"Reference : {response.reference}")
    print(f"Document Class : {response.classification.document_type} [{response.classification.jurisdiction}]")
    print(f"Status    : {response.status.value}")
    print(f"Reason    : {response.reason}")
    print(f"Attempts  : {response.attempt_count}")
    print(f"Results   : {len(response.results)}")
    for index, result in enumerate(response.results, start=1):
        print(f"\nResult {index}")
        print(f"  Title       : {result.title}")
        print(f"  Source      : {result.source}")
        print(f"  Document ID : {result.doc_id}")
        print(f"  URL         : {result.url}")
        print(f"  Retrieved   : {result.retrieved_at.isoformat()}")
        print(f"  Metadata    : {result.metadata}")
    if response.proposed_guideline_lesson:
        print(f"\nProposed lesson: {response.proposed_guideline_lesson}")
    if full:
        print("\n" + "=" * 80)
        print("FULL RESPONSE STRUCTURE")
        print("=" * 80)
        print(json.dumps(response.model_dump(mode="json"), indent=2, ensure_ascii=False))

## Document cache and service orchestration

`DocumentService` performs extraction, canonical cache lookup, per-reference resolution, and aggregation. The cache is reused across notebook runs through the existing JSONL file.

In [ ]:
from src.document_helpers.cache import CacheStatus, DocumentCache, normalize_reference

cache = DocumentCache(agent_config.cache_path, model=agent_config.model)
print(f"Document cache configured at: {cache.path}")

## Extraction and rendering helpers

The extraction module validates mandatory character spans against normalized text. Rendering uses the exact mention text to locate corresponding HTML content.

In [ ]:
from src.document_helpers.extraction import DocumentReference, DocumentReferences, Mention, ReferenceExtractor, validate_mentions
from src.document_helpers.rendering import fetch_formatted_page, read_linked_html, write_linked_html

## Scraping helpers (ebesluitvorming.gent.be)

Discover and fetch published agenda-item (agendapunt) URLs from City of Ghent
meetings.


In [ ]:
from src.tools.ebesluitvorming import agenda_item_url
import asyncio
import re
from typing import Any
import httpx
from bs4 import BeautifulSoup
from pathlib import Path


BASE_URL = "https://ebesluitvorming.gent.be"
_MEETING_ID_RE = re.compile(r"(?:/zittingen/|^)([0-9]{2}\.[0-9]{4}\.[0-9]{4}\.[0-9]{4})")
_DOCUMENT_ID_RE = re.compile(r"\b\d{4}_\w+_\d+\b")
# Site's WAF 403s on identifying/bot-like User-Agents; mimic a real browser instead.
_BROWSER_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/129.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
    "Accept-Language": "nl-BE,nl;q=0.9,en;q=0.8",
}



def extract_meeting_id(zitting: str) -> str:
    """Extract a 4-part Ghent meeting ID (e.g. '25.1218.5714.0722') from a URL or raw ID."""
    match = _MEETING_ID_RE.search(zitting.strip())
    if match:
        return match.group(1)
    raise ValueError(
        f"Could not extract a valid Ghent meeting ID from {zitting!r}. "
        "Expected format 'YY.MMDD.XXXX.XXXX' or a URL containing '/zittingen/YY.MMDD.XXXX.XXXX'."
    )


def parse_zitting_agendapunten(
    html: str,
    meeting_id: str = "",
    *,
    only_published: bool = True,
) -> list[dict[str, Any]]:
    """Extract agenda items from City of Ghent meeting (zitting) HTML."""
    soup = BeautifulSoup(html, "html.parser")
    if not meeting_id:
        link = soup.find(
            "a", href=re.compile(r"/zittingen/([0-9]{2}\.[0-9]{4}\.[0-9]{4}\.[0-9]{4})")
        )
        if link and link.get("href"):
            match = _MEETING_ID_RE.search(link["href"])
            if match:
                meeting_id = match.group(1)

    items: list[dict[str, Any]] = []
    for li in soup.find_all("li", class_="agendapage"):
        item_id = li.get("data-meeting-item-id") or ""
        pos_el = li.find(class_="position")
        pos_text = pos_el.get_text(strip=True) if pos_el else ""
        position = int(pos_text) if pos_text.isdigit() else None

        title_el = li.find("a", class_="title")
        title = title_el.get_text(strip=True) if title_el else ""
        href = title_el.get("href") if title_el else None

        item_div = li.find(class_="item")
        item_classes = item_div.get("class", []) if item_div else []
        is_published = bool(href) and ("not-published" not in item_classes)

        if only_published and not is_published:
            continue

        item_url = (
            f"{BASE_URL}{href}"
            if href
            else (agenda_item_url(meeting_id, item_id) if is_published else None)
        )

        doc_match = _DOCUMENT_ID_RE.search(title)
        document_id = doc_match.group(0) if doc_match else None

        summary_el = li.find(class_="summary")
        summary = (
            summary_el.get_text(separator=" ", strip=True) if summary_el else None
        )

        items.append(
            {
                "agenda_item_id": item_id,
                "meeting_id": meeting_id,
                "position": position,
                "title": title,
                "url": item_url,
                "is_published": is_published,
                "document_id": document_id,
                "summary": summary,
            }
        )

    return items


async def fetch_agendapunten_urls(
    url: str = "https://ebesluitvorming.gent.be/zittingen",
    *,
    month: int | str | None = None,
    year: int | str | None = None,
    client: httpx.AsyncClient | None = None,
) -> list[str]:
    """Fetch all published agendapunten URLs from https://ebesluitvorming.gent.be/zittingen.

    Parameters
    ----------
    url
        Either "https://ebesluitvorming.gent.be/zittingen" (or "/zittingen/lijst") to discover
        all meetings in the overview, OR a specific meeting URL/ID to scrape only that meeting.
    month, year
        Optional month and year when scraping the zittingen list (e.g. month=8, year=2026).
    client
        Optional httpx.AsyncClient to reuse.

    Returns
    -------
    list[str]
        Public URLs of all published agenda items found.
    """
    own_client = client is None
    c = client or httpx.AsyncClient(
        timeout=30,
        follow_redirects=True,
        headers=_BROWSER_HEADERS,
    )
    try:
        meeting_match = _MEETING_ID_RE.search(url.strip())
        if meeting_match:
            meeting_id = meeting_match.group(1)
            meeting_url = f"{BASE_URL}/zittingen/{meeting_id}"
            res = await c.get(meeting_url)
            res.raise_for_status()
            soup = BeautifulSoup(res.text, "html.parser")
            urls = []
            for li in soup.find_all("li", class_="agendapage"):
                title_el = li.find("a", class_="title")
                item_div = li.find(class_="item")
                is_pub = (
                    title_el
                    and title_el.get("href")
                    and ("not-published" not in (item_div.get("class", []) if item_div else []))
                )
                if is_pub:
                    urls.append(f"{BASE_URL}{title_el['href']}")
            return urls

        target_url = url.strip()
        if target_url.rstrip("/").endswith("/zittingen"):
            target_url = f"{BASE_URL}/zittingen/lijst"
        if month and year and "month=" not in target_url:
            sep = "&" if "?" in target_url else "?"
            target_url = f"{target_url}{sep}month={int(month):02d}&year={year}"

        res = await c.get(target_url)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, "html.parser")
        meeting_hrefs = list(
            dict.fromkeys(
                a["href"]
                for a in soup.find_all("a", href=True)
                if a["href"].startswith("/zittingen/")
                and a["href"] not in ("/zittingen/lijst", "/zittingen/kalender")
                and "?" not in a["href"]
            )
        )

        async def _fetch_meeting(href: str) -> list[str]:
            try:
                m_res = await c.get(f"{BASE_URL}{href}")
                if m_res.status_code != 200:
                    return []
                m_soup = BeautifulSoup(m_res.text, "html.parser")
                m_urls = []
                for li in m_soup.find_all("li", class_="agendapage"):
                    title_el = li.find("a", class_="title")
                    item_div = li.find(class_="item")
                    is_pub = (
                        title_el
                        and title_el.get("href")
                        and ("not-published" not in (item_div.get("class", []) if item_div else []))
                    )
                    if is_pub:
                        m_urls.append(f"{BASE_URL}{title_el['href']}")
                return m_urls
            except Exception:
                return []

        all_results = await asyncio.gather(*[_fetch_meeting(h) for h in meeting_hrefs])
        return [item_url for sub in all_results for item_url in sub]
    finally:
        if own_client:
            await c.aclose()


def scrape_agendapunten(
    zitting: str = "https://ebesluitvorming.gent.be/zittingen",
    *,
    month: int | str | None = None,
    year: int | str | None = None,
    only_published: bool = True,
    urls_only: bool = False,
    client: httpx.Client | None = None,
) -> list[dict[str, Any]] | list[str]:
    """Scrape agendapunten from a City of Ghent meeting or zittingen overview."""
    stripped = zitting.strip()
    if stripped.startswith("<"):
        items = parse_zitting_agendapunten(stripped, only_published=only_published)
    elif stripped.rstrip("/").endswith("/zittingen") or "/zittingen/lijst" in stripped:
        urls = asyncio.run(fetch_agendapunten_urls(stripped, month=month, year=year))
        if urls_only:
            return urls
        return [{"url": u} for u in urls]
    else:
        meeting_id = extract_meeting_id(stripped)
        url = f"{BASE_URL}/zittingen/{meeting_id}"
        own_client = client is None
        http_client = client or httpx.Client(
            timeout=30,
            follow_redirects=True,
            headers=_BROWSER_HEADERS,
        )
        try:
            response = http_client.get(url)
            response.raise_for_status()
            html = response.text
        finally:
            if own_client:
                http_client.close()

        items = parse_zitting_agendapunten(
            html, meeting_id=meeting_id, only_published=only_published
        )

    if urls_only:
        return [it["url"] for it in items if it.get("url")]
    return items


async def async_scrape_agendapunten(
    zitting: str = "https://ebesluitvorming.gent.be/zittingen",
    *,
    month: int | str | None = None,
    year: int | str | None = None,
    only_published: bool = True,
    urls_only: bool = False,
    client: httpx.AsyncClient | None = None,
) -> list[dict[str, Any]] | list[str]:
    """Asynchronously scrape agendapunten from a City of Ghent meeting or zittingen overview."""
    stripped = zitting.strip()
    if stripped.startswith("<"):
        items = parse_zitting_agendapunten(stripped, only_published=only_published)
    elif stripped.rstrip("/").endswith("/zittingen") or "/zittingen/lijst" in stripped:
        urls = await fetch_agendapunten_urls(
            stripped, month=month, year=year, client=client
        )
        if urls_only:
            return urls
        return [{"url": u} for u in urls]
    else:
        meeting_id = extract_meeting_id(stripped)
        url = f"{BASE_URL}/zittingen/{meeting_id}"
        own_client = client is None
        http_client = client or httpx.AsyncClient(
            timeout=30,
            follow_redirects=True,
            headers=_BROWSER_HEADERS,
        )
        try:
            response = await http_client.get(url)
            response.raise_for_status()
            html = response.text
        finally:
            if own_client:
                await http_client.aclose()

        items = parse_zitting_agendapunten(
            html, meeting_id=meeting_id, only_published=only_published
        )

    if urls_only:
        return [it["url"] for it in items if it.get("url")]
    return items


In [ ]:
import asyncio
from collections import Counter
from tqdm import tqdm

processed = []


def print_document_summary(url: str, result):
    status_counts = Counter(
        outcome.resolution.status.value if outcome.resolution else "error"
        for outcome in result.references
    )
    counts = ", ".join(
        f"{status}={status_counts[status]}"
        for status in ("resolved", "ambiguous", "unresolved", "error")
        if status_counts[status]
    ) or "none"
    print(f"\nProcessed {url}")
    print(f"  document={result.status.value} | references={len(result.references)} | {counts}")
    for index, outcome in enumerate(result.references, start=1):
        reference = outcome.extracted.doc_reference
        if outcome.resolution:
            response = outcome.resolution
            detail = response.results[0].url if response.results else response.reason.strip()
            cache = outcome.cache_status.value if outcome.cache_status else "n/a"
            print(f"  {index:>2}. {response.status.value:<10} cache={cache:<4} {reference} -> {detail}")
        else:
            print(f"  {index:>2}. error      {reference} -> {outcome.error}")
    if result.errors:
        print(f"  errors: {len(result.errors)}")


async def process_url(url: str):
    html, text = fetch_formatted_page(url)
    result = await document_service.process(
        text,
        document_context=DocumentContext(
            municipality="Ghent, Flanders, Belgium",
            description="Extracted from a Ghent agenda item.",
        ),
    )
    print_document_summary(url, result)
    return html, text, result

## Single-URL showcase

Fetch one agenda-item URL, extract and resolve its references, then write and
display the linked HTML.


In [ ]:
urls = await fetch_agendapunten_urls(
    "https://ebesluitvorming.gent.be/zittingen", month=8, year=2026
)


urls

In [ ]:
url = "https://ebesluitvorming.gent.be/zittingen/23.0216.8231.4913/agendapunten/23.1027.3232.6919"

html, text, document_result = await process_url(url)
processed = document_result.references

In [ ]:
filter_unresolved = [outcome for outcome in processed if not outcome.resolution.status.value == "resolved"]
for outcome in filter_unresolved:
    print(f"\nReference: {outcome.extracted.doc_reference}")
    if outcome.resolution:
        format_resolution(outcome.resolution)
    else:
        print(f"Resolution error: {outcome.error}")

In [ ]:
for mention in document_result.references[-3].extracted.mentions:
    print(mention)

In [ ]:
from IPython.display import HTML, display
item_id = url.split("/")[-1]
output_file = Path(f"results/agenda_item_linked-{item_id}.html")
write_linked_html(html, document_result.references, output_file)
print(f"Linked HTML written to {output_file}")
#display(HTML(output_file.read_text(encoding="utf-8")))

## Batch showcase

Run the pipeline over many agenda items discovered from the zittingen overview,
writing one linked HTML file per item into `results/`.


In [ ]:
urls = await fetch_agendapunten_urls("https://ebesluitvorming.gent.be/zittingen", month=10, year=2026)
print(f"Total published agendapunten URLs found: {len(urls)}")
failures = 0
selected_urls = urls[0:400]
total_items = len(selected_urls)
for idx, url in enumerate(selected_urls, start=1):
    item_id = url.split("/")[-1]
    prev_output_file_b1 = Path(f"results/batch_1/agenda_item_linked-{item_id}.html")
    prev_output_file_b2 = Path(f"results/batch_2/agenda_item_linked-{item_id}.html")
    if prev_output_file_b1.exists() or prev_output_file_b2.exists():
        print(f"  skipped: {prev_output_file_b1} or {prev_output_file_b2} already exists")
        continue
    print(f"\n[{idx}/{total_items}]")
    try:
        html, text, result = await process_url(url)
        
        output_file = Path(f"results/batch_2/agenda_item_linked-{item_id}.html")
        write_linked_html(html, result.references, output_file)
        print(f"  saved: {output_file}")
    except Exception as error:
        failures += 1
        print(f"  failed: {url}.")
print(f"Total failures: {failures}")

## Resolution spot-check

Resolve a single reference end-to-end to inspect the raw `ResolutionResponse`.


In [ ]:
ghent_document_context = DocumentContext(
    municipality="Ghent",
    description=(
        "Extracted from documents from Ghent. Use Ghent, Flemish regional law, "
        "and Belgian federal law as fallback context only."
    ),
)

query = "Wet van 7 mei 1999 betreffende de kansspelen, de kansspelinrichtingen en de bescherming van de spelers"


results = await find_link_to_document(
    query,
    document_context=ghent_document_context,
)
format_resolution(results)

In [ ]:
results.reference

# Statistics

In [ ]:
from collections import Counter
from bs4 import BeautifulSoup


def summarize_stored_results(results_dir: str | Path = "results/batch_2"):
    result_files = sorted(Path(results_dir).rglob("agenda_item_linked-*.html"))
    status_counts = Counter()
    unique_references = set()
    unique_unresolved_references = {}
    unique_ambiguous_references = {}
    unresolved_references = []
    marking_failures = []
    errors = []
    references_found = 0

    for result_file in result_files:
        try:
            html, entries = read_linked_html(result_file)
            soup = BeautifulSoup(html, "html.parser")
            reference_statuses = {}
            for entry in entries:
                reference = entry["reference"].doc_reference
                response = entry["results"]
                status = response.status.value
                current_status = reference_statuses.get(reference)
                if current_status is None or status == "resolved":
                    reference_statuses[reference] = status

            for reference, status in reference_statuses.items():
                normalized_reference = reference.casefold()
                references_found += 1
                unique_references.add(normalized_reference)
                status_counts[status] += 1
                if status == "ambiguous":
                    unique_ambiguous_references.setdefault(normalized_reference, reference)
                elif status == "unresolved":
                    unique_unresolved_references.setdefault(normalized_reference, reference)
                if status in {"ambiguous", "unresolved"}:
                    unresolved_references.append(
                        {"file": result_file.name, "reference": reference, "status": status}
                    )

            warning = soup.select_one("aside.legal-reference-marking-warning")
            if warning:
                for item in warning.select("li"):
                    marking_failures.append(
                        {"file": result_file.name, "detail": item.get_text(" ", strip=True)}
                    )
        except Exception as error:
            errors.append({"file": result_file.name, "error": str(error)})

    return {
        "files_found": len(result_files),
        "files_processed": len(result_files) - len(errors),
        "files_failed": len(errors),
        "references_found": references_found,
        "unique_references": len(unique_references),
        "unique_ambiguous": len(unique_ambiguous_references),
        "unique_unresolved": len(unique_unresolved_references),
        "unique_ambiguous_references": sorted(unique_ambiguous_references.values(), key=str.casefold),
        "unique_unresolved_references": sorted(unique_unresolved_references.values(), key=str.casefold),
        "resolved": status_counts["resolved"],
        "ambiguous": status_counts["ambiguous"],
        "unresolved": status_counts["unresolved"],
        "marking_failures": len(marking_failures),
        "unresolved_references": unresolved_references,
        "marking_failure_details": marking_failures,
        "errors": errors,
    }


def print_stored_results_overview(results_dir: str | Path = "results/batch_2"):
    stats = summarize_stored_results(results_dir)
    print("Stored results overview")
    print(f"  Files found                    : {stats['files_found']}")
    print(f"  Files processed                : {stats['files_processed']}")
    print(f"  Files failed                   : {stats['files_failed']}")
    print(f"  References found               : {stats['references_found']}")
    print(f"  Unique references              : {stats['unique_references']}")
    print(f"  Resolved occurrences           : {stats['resolved']}")
    print(f"  Ambiguous occurrences          : {stats['ambiguous']}")
    print(f"  Unique ambiguous references    : {stats['unique_ambiguous']}")
    print(f"  Unresolved occurrences         : {stats['unresolved']}")
    print(f"  Unique unresolved references   : {stats['unique_unresolved']}")
    print(f"  Marking failures               : {stats['marking_failures']}")

    if stats["unique_unresolved_references"]:
        print("Unique unresolved references:")
        for reference in stats["unique_unresolved_references"]:
            print(f"  - {reference}")
    if stats["unique_ambiguous_references"]:
        print("Unique ambiguous references:")
        for reference in stats["unique_ambiguous_references"]:
            print(f"  - {reference}")
    if stats["marking_failure_details"]:
        print("References not marked in stored HTML:")
        for item in stats["marking_failure_details"]:
            print(f"  - {item['file']}: {item['detail']}")
    if stats["errors"]:
        print("Files that could not be read:")
        for item in stats["errors"]:
            print(f"  - {item['file']}: {item['error']}")
    return stats


stored_results_stats = print_stored_results_overview(results_dir="results/batch_2")

In [ ]:
unique_references = list(set(item['reference'] for item in stored_results_stats['unresolved_references']))
unique_docs = list(set(item['file'] for item in stored_results_stats['unresolved_references']))
print(f"Found {len(unique_references)} unique unresolved references")
print(f"Found {len(unique_docs)} unique unresolved documents")

In [ ]:
stored_results_stats

In [ ]:

for index, reference in enumerate(unique_docs):
    print(index, reference)

In [ ]:
from collections import Counter, defaultdict


ghent_document_context = DocumentContext(
    municipality="Ghent",
    description=(
        "Extracted from documents from Ghent. Use Ghent, Flemish regional law, "
        "and Belgian federal law as fallback context only."
    ),
)

recheck_outcomes = []
for ref in unique_references[:]:
    print("\n----------------------------")
    print(f"Reference: {ref}")

    response = await find_link_to_document(
        reference=ref,
        document_context=ghent_document_context,
    )
    outcome = {
        "reference": ref,
        "status": response.status.value,
        "reason": response.reason.strip() or "No reason provided",
        "attempts": response.attempt_count,
        "candidate_count": len(response.results),
        "classification": response.classification.model_dump(mode="json") if response.classification else None,
        "results": [
            {
                "title": result.title,
                "url": result.url,
                "source": result.source,
                "doc_id": result.doc_id,
                "retrieved_at": result.retrieved_at.isoformat(),
                "metadata": result.metadata,
            }
            for result in response.results
        ],
    }
    recheck_outcomes.append(outcome)

    print(f"Status: {response.status.value}")
    print(f"Reason: {response.reason}")
    print(f"Attempts: {response.attempt_count}")
    if response.classification:
        print(
            "Classification: "
            f"{response.classification.document_type} "
            f"[{response.classification.jurisdiction}]"
        )
    if response.results:
        print("Results:")
        for index, result in enumerate(response.results, start=1):
            print(f"  Result {index}")
            print(f"    Title       : {result.title}")
            print(f"    Source      : {result.source}")
            print(f"    Document ID : {result.doc_id}")
            print(f"    URL         : {result.url}")
    else:
        print("Results: none")

In [ ]:
status_counts = Counter(outcome["status"] for outcome in recheck_outcomes)
reason_counts = Counter(outcome["reason"] for outcome in recheck_outcomes)
references_by_reason = defaultdict(list)
for outcome in recheck_outcomes:
    references_by_reason[outcome["reason"]].append(outcome["reference"])

non_resolved_count = sum(outcome["status"] != "resolved" for outcome in recheck_outcomes)





no_candidate_count = sum(outcome["candidate_count"] == 0 for outcome in recheck_outcomes)
print("\nRecheck analysis")
print(f"  References checked : {len(recheck_outcomes)}")
print(f"  Resolved           : {status_counts['resolved']}")
print(f"  Ambiguous          : {status_counts['ambiguous']}")
print(f"  Unresolved         : {status_counts['unresolved']}")
print(f"  Still non-resolved : {non_resolved_count}")
print(f"  Without candidates : {no_candidate_count}")
print("Reasons:")
for reason, count in reason_counts.most_common():
    print(f"  [{count}] {reason}")
    for reference in references_by_reason[reason]:
        print(f"      - {reference}")

In [ ]:
from collections import Counter, defaultdict



non_resolved_outcomes = [

    outcome

    for outcome in recheck_outcomes

    if outcome["status"] in {"unresolved", "ambiguous"}

]



by_document_type = Counter(

    (outcome["classification"] or {}).get("document_type") or "unclassified"

    for outcome in non_resolved_outcomes

)

by_jurisdiction = Counter(

    (outcome["classification"] or {}).get("jurisdiction") or "unknown"

    for outcome in non_resolved_outcomes

)

by_source = Counter(

    (outcome["classification"] or {}).get("source") or "no_source"

    for outcome in non_resolved_outcomes

)

by_type_and_status = Counter(

    (

        (outcome["classification"] or {}).get("document_type") or "unclassified",

        outcome["status"],

    )

    for outcome in non_resolved_outcomes

)

examples_by_type = defaultdict(list)

for outcome in non_resolved_outcomes:

    document_type = (outcome["classification"] or {}).get("document_type") or "unclassified"

    examples_by_type[document_type].append(outcome["reference"])



def non_resolved_subtype(outcome):

    reference = outcome["reference"].casefold()

    classification = outcome["classification"] or {}

    if classification.get("source") is None:

        return "unsupported or insufficiently classified"

    if classification.get("document_type") == "local_decision":

        if "deputatie" in reference:

            return "provincial or non-Ghent permit decision"

        if "omv_" in reference or "omgevingsvergunning" in reference:

            return "modern environmental-permit decision"

        if any(term in reference for term in (

            "stedenbouwkundige vergunning", "verkavelingsvergunning", "bouwvergunning",

            "stedenbouwkundige weigering", "weigering ",

        )):

            return "legacy planning or subdivision decision"

        return "other Ghent local decision"

    if classification.get("document_type") == "regulation":

        if "politiereglement" in reference:

            return "historic municipal regulation/version"

        if "waterverkoop" in reference:

            return "water-sale regulation/version"

        if "code van goede praktijk" in reference:

            return "technical guidance or code"

        return "other regulation"

    return classification.get("document_type") or "unclassified"



subtype_counts = Counter(non_resolved_subtype(outcome) for outcome in non_resolved_outcomes)

signal_counts = Counter()

for outcome in non_resolved_outcomes:

    reason = outcome["reason"].casefold()

    if "truncated" in reason or "afgekapt" in reason:

        signal_counts["truncated result set"] += 1

    if "no candidates were returned" in reason or "geen kandidaten" in reason:

        signal_counts["zero candidate pool"] += 1

    if "source" in reason and ("not establish" in reason or "geen ondersteunde bron" in reason):

        signal_counts["unsupported or missing source"] += 1

    if "candidate validation failed" in reason:

        signal_counts["validator contract failure"] += 1



print("Non-resolved recheck outcomes by document type")

for document_type, total in by_document_type.most_common():

    unresolved = by_type_and_status[document_type, "unresolved"]

    ambiguous = by_type_and_status[document_type, "ambiguous"]

    print(f"  {document_type}: {total} ({unresolved} unresolved, {ambiguous} ambiguous)")

    for reference in examples_by_type[document_type]:

        print(f"    - {reference}")



print("\nBy jurisdiction:", dict(by_jurisdiction.most_common()))

print("By source:", dict(by_source.most_common()))

print("By subtype:", dict(subtype_counts.most_common()))

print("Observed failure signals (non-exclusive):", dict(signal_counts.most_common()))
